# Advanced Neural Architectures for Touch Detection (`model_advanced.ipynb`)

This notebook trains advanced deep neural architectures (**ResNet1D** and **TouchAttentionNet**) using the **full 16 features per timestep** ($8 \text{ spatial coordinates} + 8 \text{ velocities}$) across $5 \text{ timesteps}$ ($N, 5, 16$).

## 1. Imports & Hyperparameters Setup

In [1]:
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report

# Set random seeds
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# Hyperparameters
SEQ_LEN = 5          # 5 frame timesteps
FEATURE_DIM = 16     # 8 coordinates + 8 velocities per timestep
BATCH_SIZE = 32
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-4
EPOCHS = 40

# Setup device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Load & Normalize Datasets (5 Timesteps x 16 Features)

In [2]:
def extract_5step_features_16dim(csv_path):
    df = pd.read_csv(csv_path)
    n_samples = len(df)
    
    X = np.zeros((n_samples, 5, 16), dtype=np.float32)
    
    for k in range(1, 6):
        coord_cols = [
            f"wrist{k}_x", f"wrist{k}_y",
            f"mcp{k}_x", f"mcp{k}_y",
            f"pip{k}_x", f"pip{k}_y",
            f"dip{k}_x", f"dip{k}_y"
        ]
        coords_vals = df[coord_cols].fillna(0.0).values.astype(np.float32)
        
        if k == 1:
            vel_vals = np.zeros((n_samples, 8), dtype=np.float32)
        else:
            v_idx = k - 1
            vel_cols = [
                f"wrist{v_idx}_vx", f"wrist{v_idx}_vy",
                f"mcp{v_idx}_vx", f"mcp{v_idx}_vy",
                f"pip{v_idx}_vx", f"pip{v_idx}_vy",
                f"dip{v_idx}_vx", f"dip{v_idx}_vy"
            ]
            vel_vals = df[vel_cols].fillna(0.0).values.astype(np.float32)
            
        step_vals = np.hstack([coords_vals, vel_vals])
        X[:, k - 1, :] = step_vals
        
    target_col = "touch_finger" if "touch_finger" in df.columns else "touch"
    y = df[target_col].astype(str).str.strip().str.lower().isin(["1", "true", "t", "yes", "y"]).values.astype(np.float32)
    y = y.reshape(-1, 1)
    
    return X, y

TRAIN_CSV = "./data/training_data.csv"
TEST_CSV = "./data/test_data.csv"

X_train_np, y_train_np = extract_5step_features_16dim(TRAIN_CSV)
X_test_np, y_test_np = extract_5step_features_16dim(TEST_CSV)

# Normalize using StandardScaler
scaler = StandardScaler()
N_tr, T, C = X_train_np.shape
N_te, _, _ = X_test_np.shape

X_train_flat = X_train_np.reshape(N_tr, -1)
X_test_flat = X_test_np.reshape(N_te, -1)

X_train_scaled = scaler.fit_transform(X_train_flat).reshape(N_tr, T, C)
X_test_scaled = scaler.transform(X_test_flat).reshape(N_te, T, C)

X_train_tensor = torch.from_numpy(X_train_scaled).type(torch.float32)
y_train_tensor = torch.from_numpy(y_train_np).type(torch.float32)
X_test_tensor = torch.from_numpy(X_test_scaled).type(torch.float32)
y_test_tensor = torch.from_numpy(y_test_np).type(torch.float32)

print(f"Normalized X_train shape (5 steps x 16 features): {X_train_tensor.shape}, y_train shape: {y_train_tensor.shape}")
print(f"Normalized X_test shape  (5 steps x 16 features): {X_test_tensor.shape},  y_test shape:  {y_test_tensor.shape}")

## 3. Model Architecture 1: 1D Residual Network (ResNet1D)

Combines residual skip connections (`x + conv(x)`) with 1D Convolutions across the 16 channels.

In [3]:
class ResBlock1D(nn.Module):
    def __init__(self, channels: int, dropout: float = 0.2):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv1d(channels, channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(channels),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(channels, channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(channels)
        )
        self.relu = nn.ReLU()
        
    def forward(self, x):
        return self.relu(x + self.block(x))

class FingerTouchResNet1D(nn.Module):
    def __init__(self, in_channels: int = 16, hidden_dim: int = 64, dropout: float = 0.2):
        super().__init__()
        self.input_conv = nn.Sequential(
            nn.Conv1d(in_channels, hidden_dim, kernel_size=3, padding=1),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU()
        )
        self.res1 = ResBlock1D(hidden_dim, dropout)
        self.res2 = ResBlock1D(hidden_dim, dropout)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(hidden_dim, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1)
        )
        
    def forward(self, x):
        x_conv = x.permute(0, 2, 1)
        out = self.input_conv(x_conv)
        out = self.res1(out)
        out = self.res2(out)
        out = self.pool(out)
        return self.classifier(out)

resnet1d = FingerTouchResNet1D(in_channels=FEATURE_DIM, hidden_dim=64).to(device)
print(resnet1d)

## 4. Model Architecture 2: TouchAttentionNet (Multi-Head Self-Attention)

Computes dynamic spatial-temporal self-attention across the 16 features.

In [4]:
class TouchAttentionNet(nn.Module):
    def __init__(self, input_dim: int = 16, embed_dim: int = 32, num_heads: int = 4, dropout: float = 0.2):
        super().__init__()
        self.embedding = nn.Linear(input_dim, embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, embed_dim)
        )
        self.norm2 = nn.LayerNorm(embed_dim)
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )
        
    def forward(self, x):
        emb = self.embedding(x)
        attn_out, _ = self.attn(emb, emb, emb)
        x_attn = self.norm1(emb + attn_out)
        ffn_out = self.ffn(x_attn)
        x_out = self.norm2(x_attn + ffn_out)
        pooled = x_out.mean(dim=1)
        return self.classifier(pooled)

attn_net = TouchAttentionNet(input_dim=FEATURE_DIM, embed_dim=32, num_heads=4).to(device)
print(attn_net)

## 5. Training PyTorch Neural Architectures

In [5]:
train_loader = DataLoader(list(zip(X_train_tensor, y_train_tensor)), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(list(zip(X_test_tensor, y_test_tensor)), batch_size=BATCH_SIZE, shuffle=False)

def train_model(model, epochs=40, lr=0.001):
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
    
    def accuracy_fn(y_true, y_pred):
        return (torch.eq(y_true, y_pred).sum().item() / len(y_pred)) * 100.0
        
    for epoch in range(1, epochs + 1):
        model.train()
        tr_loss, tr_acc = 0.0, 0.0
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            logits = model(X_b)
            loss = loss_fn(logits, y_b)
            preds = torch.round(torch.sigmoid(logits))
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            tr_loss += loss.item() * len(X_b)
            tr_acc += accuracy_fn(y_b, preds) * len(X_b) / 100.0
            
        model.eval()
        te_loss, te_acc = 0.0, 0.0
        with torch.inference_mode():
            for X_b, y_b in test_loader:
                X_b, y_b = X_b.to(device), y_b.to(device)
                logits = model(X_b)
                loss = loss_fn(logits, y_b)
                preds = torch.round(torch.sigmoid(logits))
                te_loss += loss.item() * len(X_b)
                te_acc += accuracy_fn(y_b, preds) * len(X_b) / 100.0
                
        tr_acc = (tr_acc / len(X_train_tensor)) * 100.0
        te_acc = (te_acc / len(X_test_tensor)) * 100.0
        te_loss /= len(X_test_tensor)
        scheduler.step(te_loss)
        
        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoch: {epoch:02d} | Train Acc: {tr_acc:.2f}% | Test Loss: {te_loss:.4f} | Test Acc: {te_acc:.2f}%")
            
    return te_acc

print("\n=== Training ResNet1D (16 features x 5 steps) ===")
resnet_acc = train_model(resnet1d, epochs=EPOCHS)

print("\n=== Training TouchAttentionNet (16 features x 5 steps) ===")
attn_acc = train_model(attn_net, epochs=EPOCHS)

## 6. Confusion Matrices & Heatmaps & Classification Reports

In [6]:
def print_eval(model, name):
    model.eval()
    all_preds, all_targets = [], []
    with torch.inference_mode():
        for X_b, y_b in test_loader:
            X_b = X_b.to(device)
            logits = model(X_b)
            preds = torch.round(torch.sigmoid(logits)).cpu().numpy()
            all_preds.extend(preds)
            all_targets.extend(y_b.numpy())
            
    all_preds = np.array(all_preds).squeeze()
    all_targets = np.array(all_targets).squeeze()
    cm = confusion_matrix(all_targets, all_preds)
    labels = ["Untouch (0)", "Touch (1)"]
    
    # Printed Text Analytics
    print("\n" + "="*50)
    print(f" CONFUSION MATRIX ({name})")
    print("="*50)
    print(cm)
    print("\nClassification Report:\n", classification_report(all_targets, all_preds, target_names=labels))
    print("="*50)
    
    # Visual Heatmap
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels)
    plt.title(f"Confusion Matrix Heatmap - {name}")
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.tight_layout()
    plt.show()

print_eval(resnet1d, "ResNet1D")
print_eval(attn_net, "TouchAttentionNet")

## 7. Model Performance Summary & Saving Models

In [7]:
print("\n" + "="*60)
print(" NEURAL ARCHITECTURES (16 FEATURES x 5 STEPS) PERFORMANCE")
print("="*60)
print(f"  ResNet1D (Skip CNN):   {resnet_acc:.2f}%")
print(f"  TouchAttentionNet:     {attn_acc:.2f}%")
print("="*60)

torch.save(resnet1d.state_dict(), "finger_touch_resnet1d.pth")
torch.save(attn_net.state_dict(), "finger_touch_attention.pth")
print("Saved model weights to 'finger_touch_resnet1d.pth' and 'finger_touch_attention.pth'.")